In [ ]:
import pandas as pd
import scipy
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, roc_auc_score
import torch
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoConfig

import emoji
import re
import datetime


from transformers import pipeline
from tqdm import tqdm

from scipy.special import softmax

# Tweets Data

In [2]:
all_tweets = pd.read_csv("../Dataset/bitcoin-tweets-2021.csv", encoding='utf-8', lineterminator='\n', low_memory=False)
all_tweets.head()

,datetime,username,text
0,2021-01-01 23:59:58+00:00,WoodooProd,Crypto adoption in 2021: Top trends and predic...
1,2021-01-01 23:59:53+00:00,YSojourners,Ripple Executive Says US Laws on Crypto Are Va...
2,2021-01-01 23:59:53+00:00,Andama_n,@hextoysofficial | https://t.co/C6LPpuq05n Are...
3,2021-01-01 23:59:51+00:00,YSojourners,FUN and Games: Altcoin Drastically Pumps And D...
4,2021-01-01 23:59:50+00:00,HunnerSa,I bought #Bitcoin at 6k-8k-9k-12k-16k-19k and ...


In [ ]:
def tweet_period(df, ):
    df['postdate'] = pd.to_datetime(df['datetime']).dt.date
    df = df[(df['postdate'] < datetime.date(2021, 8, 1)) & (df['postdate'] > datetime.date(2021, 2, 1))].copy()
    df['posttime'] = pd.to_datetime(df['datetime']).dt.time
    
    return df

# relevant tweets for decided time period 

rel_tweets = tweet_period(all_tweets) # 8230219
# rel_tweets

,datetime,username,text,postdate,posttime
1415157,2021-02-02 23:59:57+00:00,scalperSignals,#LTCDOWN/#USDT\nDetails and advanced trading t...,2021-02-02,23:59:57
1415158,2021-02-02 23:59:44+00:00,DerpieSloth,@PriceSmith8 @WSBChairman So $LINK is proven t...,2021-02-02,23:59:44
1415159,2021-02-02 23:59:43+00:00,xbmcvlogs,The amount of people that have no idea about #...,2021-02-02,23:59:43
1415160,2021-02-02 23:59:43+00:00,I_Wicklund,@alex_remsberg @SkAlecsZ @WikiOoc Yeah it stan...,2021-02-02,23:59:43
1415161,2021-02-02 23:59:43+00:00,cryptoh46184684,@nvk #Bitcoin goes up again wen #altcoins are ...,2021-02-02,23:59:43
...,...,...,...,...,...
9645371,2021-07-31 00:00:01+00:00,Telugu_HODL,"@ZeroHedge_ It all started géttimg better, aft...",2021-07-31,00:00:01
9645372,2021-07-31 00:00:00+00:00,hodlffs,LEEET'S GOOO BAAABY! #bitcoin #BTC #HodlFFS ht...,2021-07-31,00:00:00
9645373,2021-07-31 00:00:00+00:00,CryptoNerdApp,Current Price of Bitcoin:\n$41979 (4.99%)\n\n$...,2021-07-31,00:00:00
9645374,2021-07-31 00:00:00+00:00,TheCHIEF_IN,THIS IS WHY #Bitcoin https://t.co/PKERc6L9gK,2021-07-31,00:00:00


In [ ]:
def first_day(df, selecteddate):
    df = df[df['datetime'].str.contains(selecteddate)].copy()
    df['postdate'] = pd.to_datetime(df['datetime']).dt.date
    df['posttime'] = pd.to_datetime(df['datetime']).dt.time
    return df

firstday_tweets = first_day(all_tweets, '2021-02-10') # 60353 tweets
firsthour_tweets = firstday_tweets[firstday_tweets['posttime'] < datetime.time(1, 0 , 0)] # 1440 tweets

In [ ]:
def preprocess_emoji_test(text):
    if pd.isna(text):
        return ""

    # convert emoji to text 
    text = emoji.demojize(text, language='en')

    # remove urls 
    text = re.sub(r"http\S+|www\.\S+", "", text)

    # remove spaces
    text = " ".join(text.split())

    # lowercase
    text = text.lower()

    return text

firsthour_tweets = firsthour_tweets.assign(c_text = firsthour_tweets["text"].apply(preprocess_emoji_test).copy())

# BTC Data

In [18]:
btc = pd.read_csv("../Dataset/btcusd_1-min_data.csv")
btc['datetime'] = pd.to_datetime(btc['Timestamp'],unit='s')
btc['date'] = pd.to_datetime(btc['datetime']).dt.date
btc['time'] = pd.to_datetime(btc['datetime']).dt.time

In [ ]:
# relevant bitcoin data for selected days
rel_btc = btc[(btc['date'] < datetime.date(2021, 8, 1)) & (btc['date'] > datetime.date(2021, 2, 1))].copy() # 259200

In [19]:
sameday_btc = btc[btc['date'] == datetime.date(2021, 2, 10)]
# sameday_btc
btc_d1h1 = sameday_btc[sameday_btc['time'] < datetime.time(1, 0, 0)]

In [20]:
btc_d1h1.head()

,Timestamp,Open,High,Low,Close,Volume,datetime,date,time
4791719,1.612915e+09,47072.73,47094.48,47031.93,47070.92,2.580461,2021-02-10 00:00:00,2021-02-10,00:00:00
4791720,1.612915e+09,47069.71,47133.54,47069.71,47104.45,3.023258,2021-02-10 00:01:00,2021-02-10,00:01:00
4791721,1.612915e+09,47113.37,47200.00,47113.37,47156.75,13.636693,2021-02-10 00:02:00,2021-02-10,00:02:00
4791722,1.612915e+09,47158.75,47232.64,47158.75,47187.92,6.975702,2021-02-10 00:03:00,2021-02-10,00:03:00
4791723,1.612915e+09,47169.17,47216.23,47155.46,47189.08,4.717993,2021-02-10 00:04:00,2021-02-10,00:04:00


# Embeddings for 1st hour / 1 minute

In [71]:
firsthour_tweets

,datetime,username,text,postdate,posttime,c_text
1882557,2021-02-10 00:59:59+00:00,mickboylan,https://t.co/eJMhX206gp\n\nCashaa in the conve...,2021-02-10,00:59:59,cashaa in the conversation along side bitcoin ...
1882558,2021-02-10 00:59:58+00:00,Brett_Beech,@jakeb515 it's just a really bad time to get t...,2021-02-10,00:59:58,@jakeb515 it's just a really bad time to get t...
1882559,2021-02-10 00:59:57+00:00,AmerLiberal,@bernieshq WHERE are you getting your disinfor...,2021-02-10,00:59:57,@bernieshq where are you getting your disinfor...
1882560,2021-02-10 00:59:56+00:00,DHusko23,Bitcoin seems like a pretty big waste of energy,2021-02-10,00:59:56,bitcoin seems like a pretty big waste of energy
1882561,2021-02-10 00:59:56+00:00,POSTkingPiNwin,We're Gonna Need A Bigger Moon\n#WBITG1TGA #Bi...,2021-02-10,00:59:56,we're gonna need a bigger moon #wbitg1tga #bit...
...,...,...,...,...,...,...
1884521,2021-02-10 00:00:01+00:00,KCTV5,The price of Bitcoin has soared this week whil...,2021-02-10,00:00:01,the price of bitcoin has soared this week whil...
1884522,2021-02-10 00:00:01+00:00,isaacwalex,"Imagine holding #bitcoin and $MSTR, crazy ride...",2021-02-10,00:00:01,"imagine holding #bitcoin and $mstr, crazy ride..."
1884523,2021-02-10 00:00:00+00:00,jamminJKayz,"TESLA FINALLY CAME OUT = BITCOIN, AFTER ALL TH...",2021-02-10,00:00:00,"tesla finally came out = bitcoin, after all th..."
1884524,2021-02-10 00:00:00+00:00,bilpcoinbpc,Let's grow together\n\n1. Like/Retweet\n2. Lin...,2021-02-10,00:00:00,let's grow together 1. like/retweet 2. link yt...


In [24]:
d1m_tweets = firsthour_tweets[firsthour_tweets['posttime'] < datetime.time(0, 1, 0)]

In [25]:
robertaMODEL = f"cardiffnlp/twitter-roberta-base-sentiment-latest"
robertatokenizer = AutoTokenizer.from_pretrained(robertaMODEL)
config = AutoConfig.from_pretrained(robertaMODEL)
# PT
robertamodel = AutoModelForSequenceClassification.from_pretrained(robertaMODEL)
robertamodel.save_pretrained(robertaMODEL)

Some weights of the model checkpoint at cardiffnlp/twitter-roberta-base-sentiment-latest were not used when initializing RobertaForSequenceClassification: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
- This IS expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing RobertaForSequenceClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).


In [ ]:
def robert_test(data):
    X = data['c_text'].iloc[:].to_list()

    
    preds = []
    preds_proba = []
    for x in X:
        # with torch.no_grad():
        encoded_input = robertatokenizer(x, return_tensors='pt')
        logits = robertamodel(**encoded_input).logits
        output = robertamodel(**encoded_input)
        # input_sequence = tokenizer(x, return_tensors="pt", **tokenizer_kwargs)
        # logits = model(**input_sequence).logits
        scores = output[0][0].detach().numpy()
        scores = softmax(scores)
        ranking = np.argsort(scores)
        ranking = ranking[::-1]
        sent_pred = config.id2label[ranking[0]]
        # print(x, sent_pred)

        preds.append(sent_pred)
        preds_proba.append(scores[ranking[0]])

    # data['sentiment'] = preds
    # data['sentiment_score'] = preds_proba
    data = data.assign(sentiment = preds)
    data = data.assign(sentiment = preds_proba)

    return data


In [ ]:
res2 = robert_test(d1m_tweets)

In [ ]:
resh = robert_test(firsthour_tweets) 

In [ ]:
def pminsent(df): 
    data = {}
    time = []
    likely_sentiment = []
    sent_confidence = []
    total_tweets = []
    sentiment_tweets = []
    
    for min in range(59):
        pmdf = df[(df['posttime'] < datetime.time(0, min+1, 0)) & (df['posttime'] >= datetime.time(0, min, 0))]
        agg = pmdf.groupby('sentiment')['sentiment_score'].sum()
        overall_sentiment = agg.idxmax()
        confidence = agg.max() / agg.sum()
        time.append(f"min{min+1}")
        likely_sentiment.append(overall_sentiment)
        sent_confidence.append(confidence)
        total_tweets.append(pmdf['sentiment'].count())
        sentiment_tweets.append(pmdf.groupby('sentiment')['sentiment'].count().max())

    data.update({"time": time, 
                "overall_sentiment": likely_sentiment,
                "confidence": sent_confidence,
                "total_tweets": total_tweets,
                "sentiment_tweets": sentiment_tweets,
                    })
    newdf = pd.DataFrame(data)
    
    return newdf             

In [100]:
embres = pminsent(resh)
embres.head()

,time,overall_sentiment,confidence,total_tweets,sentiment_tweets
0,min1,neutral,0.711239,55,38
1,min2,neutral,0.651111,30,20
2,min3,neutral,0.424067,32,14
3,min4,neutral,0.464534,33,15
4,min5,neutral,0.464030,31,14


# BTC prediction


In [ ]:
import plotly.express as px
fig = px.line(btc_d1h1, x="time", y="Close")
fig.show()